In [17]:
###### Using the sawgrass and mangrove data to test with Lansat signatures ######
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from rasterio.windows import from_bounds
# Read comma-separated txt file
df = pd.read_csv('/Users/bz294/Documents/WhiteZone/Validation/Sawgrass/LT_PP_Iwaniec_001.txt', sep=',')  # sep=',' tells pandas to split on commas

df['Date'] = pd.to_datetime(df['Date'])
df['Date'] = df['Date'].dt.strftime('%-m/%-d/%y')  # Works on Mac / Linux
# Show the first 5 rows
print(df)

    SITENAME     Date  Sawgrass_AG_Biomass  Sawgrass_AG_Biomass_STD  \
0     TS/Ph4  10/1/97             182.4170                  43.1571   
1     TS/Ph4  12/1/97             190.2880                  30.0164   
2     TS/Ph4   2/1/98             164.5537                  20.1016   
3     TS/Ph4   4/1/98             178.6237                  30.9346   
4     TS/Ph4   6/1/98             192.6167                  26.1123   
..       ...      ...                  ...                      ...   
107   TS/Ph5   4/1/06             124.9994                  19.1402   
108   TS/Ph5   6/1/06             153.0235                  32.1047   
109   TS/Ph5   8/1/06             159.6042                  13.2118   
110   TS/Ph5  10/1/06             148.5803                  16.3719   
111   TS/Ph5  12/1/06             132.5884                  33.1733   

     Sawgrass_Culm_Density  Sawgrass_Culm_Density_STD  
0                    35.33                       4.62  
1                    40.00         

In [18]:
# 2. Read the first CSV
csv_file1 = '/Users/bz294/Documents/WhiteZone/Validation/Sawgrass/LT_PP_Rubio_001.csv'
df_csv1 = pd.read_csv(csv_file1, sep=',', skipinitialspace=True)
print("\nRubio CSV head:")
print(df_csv1.head())

# 3. Read the second CSV
csv_file2 = '/Users/bz294/Documents/WhiteZone/Validation/Sawgrass/LT_PP_Grahl_001.csv'
df_csv2 = pd.read_csv(csv_file2, sep=',', skipinitialspace=True)
print("\nGrahl CSV head:")
print(df_csv2.head())


Rubio CSV head:
  SITENAME    Date  Sawgrass_AG_Biomass  Sawgrass_AG_Biomass_STD  \
0  TS/Ph1b  1/1/00             313.9557                 111.0977   
1  TS/Ph1b  3/1/00             228.4093                  70.2730   
2  TS/Ph1b  5/1/00             219.0292                  90.4946   
3  TS/Ph1b  7/1/00             253.9624                  96.1007   
4  TS/Ph1b  9/1/00             221.5207                  48.1614   

   Sawgrass_Culm_Density  Sawgrass_Culm_Density_STD  
0                  45.67                       3.79  
1                  43.67                       7.09  
2                  40.67                       6.66  
3                  43.67                       6.03  
4                  41.67                       3.79  

Grahl CSV head:
  SITENAME        Date  Sawgrass_AG_Biomass  Sawgrass_AG_Biomass_STD  \
0    SRS1b  2000-11-01             435.7049                  88.7635   
1     SRS2  2000-11-01             404.9774                  18.4500   
2     SRS3  2000-

In [19]:
# 4. Merge all three dataframes into one
df_merged = pd.concat([df, df_csv1, df_csv2], ignore_index=True)

# 5. Check the result
print(df_merged.head())
print(f"Total rows after merge: {len(df_merged)}")

  SITENAME     Date  Sawgrass_AG_Biomass  Sawgrass_AG_Biomass_STD  \
0   TS/Ph4  10/1/97             182.4170                  43.1571   
1   TS/Ph4  12/1/97             190.2880                  30.0164   
2   TS/Ph4   2/1/98             164.5537                  20.1016   
3   TS/Ph4   4/1/98             178.6237                  30.9346   
4   TS/Ph4   6/1/98             192.6167                  26.1123   

   Sawgrass_Culm_Density  Sawgrass_Culm_Density_STD  
0                  35.33                       4.62  
1                  40.00                       1.73  
2                  43.00                       3.61  
3                  44.00                       1.73  
4                  54.00                       9.00  
Total rows after merge: 1182


In [20]:
# Path to your Landsat time series CSV
landsat_file = '/Users/bz294/Documents/WhiteZone/Validation/Sawgrass/FCE_LTER_Landsat_TimeSeries_2000_2024_ALLSITES.csv'

# Read the CSV
df_landsat = pd.read_csv(landsat_file, sep=',', skipinitialspace=True)

# Check the first rows
print(df_landsat.head())
print(f"Total rows in Landsat file: {len(df_landsat)}")

                 system:index    Blue   Green      NDMI      NDVI      NIR  \
0  1_1_LT05_015042_19970103_0  8961.0  9461.0 -0.069703  0.122957  12439.0   
1  1_1_LT05_015042_19970103_1  8836.0  9449.0  0.009091  0.196648  14708.0   
2  1_1_LT05_015042_19970103_2  8896.0  9486.0  0.123154  0.158397  13844.0   
3  1_1_LT05_015042_19970103_3  8477.0  8882.0  0.104710  0.104484  10777.0   
4  1_1_LT05_015042_19970103_4  8473.0  8880.0  0.105705  0.224058  14043.0   

       Red      SAVI    SWIR1    SWIR2    date     sensor     site  year  \
0   9715.0  0.184432  14303.0  11186.0  1/3/97  LANDSAT_5  TS_Ph1b  1997   
1   9874.0  0.294966  14443.0  11186.0  1/3/97  LANDSAT_5   TS_Ph2  1997   
2  10058.0  0.237590  10808.0   9178.0  1/3/97  LANDSAT_5   TS_Ph3  1997   
3   8738.0  0.156722   8734.0   7987.0  1/3/97  LANDSAT_5  TS_Ph6b  1997   
4   8902.0  0.336079  11358.0   9178.0  1/3/97  LANDSAT_5   TS_Ph4  1997   

                                                .geo  
0     {"type":"Poin

In [21]:
import pandas as pd
from datetime import datetime

# -------------------------------
# Assuming you already have:
# df_merged -> merged validation data
# df_landsat -> Landsat time series CSV
# -------------------------------

# 1. Keep only relevant columns in df_merged
df_val = df_merged[['Date', 'SITENAME', 'Sawgrass_AG_Biomass']].copy()

# 2. Convert date columns to datetime
# Convert Date columns to datetime
# Example: convert validation Date column
df_val['Date'] = df_val['Date'].astype(str).str.strip()
df_val['Date'] = pd.to_datetime(df_val['Date'], infer_datetime_format=True)
# 1. Replace '/' with '_'
df_val['SITENAME'] = df_val['SITENAME'].str.replace('/', '_')

# 2. Merge "SRS1d non-burned" into "SRS1d"
df_val['SITENAME'] = df_val['SITENAME'].replace('SRS1d non-burned', 'SRS1d')

# Convert Landsat date column (assuming it's already in YYYY-MM-DD, pandas can infer)
df_landsat['date'] = pd.to_datetime(df_landsat['date'])

# 3. Filter Landsat to rows with valid NDVI
df_landsat_valid = df_landsat[df_landsat['NDVI'].notnull()].copy()

def find_closest(row, landsat_df):
    site = row['SITENAME']
    date = row['Date']
    
    # Filter Landsat to the same site
    df_site = landsat_df[landsat_df['site'] == site]
    
    if df_site.empty:
        # Return NaN for all columns if no match
        return pd.Series([pd.NaT, None] + [None]*9)
    
    df_site = df_site.copy()
    # Compute absolute date difference
    df_site['date_diff'] = (df_site['date'] - date).abs()
    
    # Pick the row with minimum difference
    closest_row = df_site.loc[df_site['date_diff'].idxmin()]
    
    # Extract bands as a flat list
    bands_list = closest_row[['Blue','Green','Red','NIR','SWIR1','SWIR2','NDVI','NDMI','SAVI']].values.tolist()
    
    return pd.Series([closest_row['date'], closest_row['date_diff'].days] + bands_list)

# Apply the function
closest_matches = df_val.apply(lambda row: find_closest(row, df_landsat_valid), axis=1)

# Assign column names
closest_matches.columns = ['Landsat_Date', 'Date_Diff_Days',
                           'Blue','Green','Red','NIR','SWIR1','SWIR2','NDVI','NDMI','SAVI']

# Merge with original validation data
df_result = pd.concat([df_val.reset_index(drop=True), closest_matches], axis=1)
df_result['Year'] = df_result['Landsat_Date'].dt.year
# Check
print(df_result.head())

# Check result
print(df_result.head())
print(f"Total rows: {len(df_result)}")

# 8. Optional: save to CSV
df_result.to_csv('/Users/bz294/Documents/WhiteZone/Validation/Sawgrass/merged_landsat_validation.csv', index=False)





/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/3097850249.py:17: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_val['Date'] = pd.to_datetime(df_val['Date'], infer_datetime_format=True)
/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/3097850249.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_val['Date'] = pd.to_datetime(df_val['Date'], infer_datetime_format=True)
/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/3097850249.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as

        Date SITENAME  Sawgrass_AG_Biomass Landsat_Date  Date_Diff_Days  \
0 1997-10-01   TS_Ph4             182.4170   1997-10-02               1   
1 1997-12-01   TS_Ph4             190.2880   1997-12-05               4   
2 1998-02-01   TS_Ph4             164.5537   1998-01-22              10   
3 1998-04-01   TS_Ph4             178.6237   1998-04-12              11   
4 1998-06-01   TS_Ph4             192.6167   1998-06-15              14   

     Blue    Green      Red      NIR    SWIR1   SWIR2      NDVI      NDMI  \
0  9135.0  10508.0  10385.0  10113.0   8315.0  7875.0 -0.013270  0.097569   
1  9624.0  10205.0  10166.0  11431.0  10859.0  9822.0  0.058573  0.025662   
2  9733.0  10312.0  10418.0  10694.0   9516.0  8848.0  0.013073  0.058288   
3  9296.0  10006.0  10160.0  10281.0   9220.0  8693.0  0.005919  0.054407   
4  8992.0   9859.0   9945.0  13321.0  10168.0  9064.0  0.145104  0.134233   

       SAVI  Year  
0 -0.019904  1997  
1  0.087857  1997  
2  0.019609  1998  
3  0.0

In [22]:
## plot - need to run the cell above first! Otherwise the number of data gets cut down ####

%matplotlib agg
# -----------------------------
# 6. Remove negative biomass
# -----------------------------
df_result = df_result[df_result['Sawgrass_AG_Biomass'] >= 0].copy()
#df_result = df_result[(df_result['Year'] > 2013)].copy()
df_result = df_result[(df_result['Year'] >= 2004) & (df_result['Year'] <= 2013)].copy()
#df_result = df_result[(df_result['Year'] < 2004)].copy()
# -----------------------------
# 7. Keep 10%-90% of each Landsat feature
# -----------------------------
features = ['Blue','Green','Red','NIR','SWIR1','SWIR2','NDVI','NDMI','SAVI']
target_col = 'Sawgrass_AG_Biomass'
target_label = 'Sawgrass AGB (g/m$^{2}$)'

for feature in features:
    q_low = df_result[feature].quantile(0.05)
    q_high = df_result[feature].quantile(0.95)
    df_result = df_result[(df_result[feature] >= q_low) & (df_result[feature] <= q_high)]

#df_result = df_result[[target] + features + ['Year']].dropna()
plt.rcParams['font.family'] = 'Times New Roman'
plt.figure(figsize=(18, 12))

for i, feature in enumerate(features):
    plt.subplot(3, 3, i+1)
    
    # Scatter points and regression line
    sns.scatterplot(x=df_result[feature], y=df_result[target_col])
    sns.regplot(x=df_result[feature], y=df_result[target_col], scatter=False, color='red')
    
    # Labels
    plt.xlabel(feature, fontsize=20)
    plt.ylabel(target_label, fontsize=20)
    
    # Reduce number of tick labels (show only 1/3)
    xticks = plt.xticks()[0]
    if feature in ['NDVI', 'NDMI', 'SAVI']:
        # Keep as float for indices
        plt.xticks(
            xticks,
            [f"{x:.2f}" if j % 3 == 0 else "" for j, x in enumerate(xticks)],
            fontsize=18
        )
    else:
        # Round to integer for bands
        plt.xticks(
            xticks,
            [f"{int(round(x))}" if j % 3 == 0 else "" for j, x in enumerate(xticks)],
            fontsize=18
        )

    yticks = plt.yticks()[0]
    # Y-axis always as integer
    plt.yticks(
        yticks,
        [f"{int(round(y))}" if j % 3 == 0 else "" for j, y in enumerate(yticks)],
        fontsize=18
    )
    
    plt.ylim(bottom=0)

    
plt.tight_layout()
plt.show()


plt.savefig(
    "/Users/bz294/Documents/WhiteZone/Validation/Sawgrass/Sawgrass_AGB_vs_Landsat_features.tif",
    format="tiff",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/206437513.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/206437513.py:77: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
## evaluating the df_val to get distribution of AGB for each site ###
out_path = "/Users/bz294/Documents/WhiteZone/Validation/Sawgrass/AGB_distribution.tif"
# Ensure Date is datetime
df_val['Date'] = pd.to_datetime(df_val['Date'], errors='coerce')

# Extract month
df_val['Month'] = df_val['Date'].dt.month

# Keep only Dec–Mar (Dec=12, Jan=1, Feb=2, Mar=3)
df_winter = df_val[df_val['Month'].isin([12, 1, 2, 3])].copy()

# Group by site and calculate median and std
stats = (
    df_winter
    .groupby('SITENAME')['Sawgrass_AG_Biomass']
    .agg(
        median_AGB='median',
        std_AGB='std',
        n_samples='count'
    )
    .reset_index()
)

print(stats)

df_winter = df_winter[df_winter['SITENAME'].isin(['TS_Ph1b','TS_Ph6b'])]


plt.figure(figsize=(6,6))

# Boxplot with only median and IQR, grey box + black median line
sns.boxplot(
    data=df_winter,
    x='SITENAME',
    y='Sawgrass_AG_Biomass',
    showfliers=False,      # remove outliers
    whiskerprops={'color':'black', 'linewidth':1.5},  # hide whiskers
    boxprops={'facecolor':'lightgrey', 'alpha':0.5},  # grey box
    medianprops={'color':'black', 'linewidth':2}  # black median line
)


plt.xlabel('Site', fontsize=20)
plt.ylabel('Sawgrass AGB (g/m$^{2}$)', fontsize=20)
plt.title('Winter Sawgrass Biomass by Site (Median + IQR)', fontsize=20)
plt.xticks(ticks=[0, 1], labels=['TS/Ph1b', 'TS/Ph6b'], fontsize=20)
plt.yticks(fontsize=20)
plt.tight_layout()

plt.savefig(
    out_path,
    format='tiff',
    dpi=300,
    bbox_inches='tight'
)
plt.show()



   SITENAME  median_AGB     std_AGB  n_samples
0     SRS1b   177.63640  138.172851          9
1     SRS1c   306.89580   42.372580          3
2     SRS1d   408.54055  166.905204         46
3      SRS2   285.72300  107.930352         49
4      SRS3   256.79790  128.764502         49
5   TS_Ph1b    88.68110   81.238832         51
6    TS_Ph2    69.24430   49.692344         51
7    TS_Ph3    82.09575   97.434286         50
8    TS_Ph4   157.05100   63.692540         19
9    TS_Ph5   174.05420   84.990168         19
10  TS_Ph6b   158.35590   57.765386         51


/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/628320.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
## Plot for two sites among the taylor slough

out_path = '/Users/bz294/Documents/WhiteZone/Validation/Sawgrass/Winter_Landsat_Boxplot_TS.tif'
# Ensure date is datetime
df_landsat_valid['date'] = pd.to_datetime(df_landsat_valid['date'], errors='coerce')

# Extract month
df_landsat_valid['Month'] = df_landsat_valid['date'].dt.month

# Winter months: Dec–Mar
df_winter = df_landsat_valid[
    df_landsat_valid['Month'].isin([12, 1, 2, 3])
].copy()

# Keep only the two sites
df_winter = df_winter[df_winter['site'].isin(['TS_Ph1b', 'TS_Ph6b'])]
features = ['Blue','Green','Red','NIR','SWIR1','SWIR2','NDVI','NDMI','SAVI']

palette = ['#1f77b4', '#ff7f0e']  # blue and orange

plt.figure(figsize=(18, 12))

for i, feature in enumerate(features):
    plt.subplot(3, 3, i + 1)
    
    # Boxplot: median and IQR only, no outliers
    sns.boxplot(
        data=df_winter,
        x='site',
        y=feature,
        showfliers=False,
        width=0.6,
        palette=palette,
        boxprops=dict(facecolor='lightgrey', alpha=0.5),
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(color='black', linewidth=1.5)
    )

    plt.xlabel('')
    plt.ylabel(feature, fontsize=20)
    plt.xticks(ticks=[0, 1], labels=['TS/Ph1b', 'TS/Ph6b'], fontsize=20)
    plt.yticks(fontsize=20)

plt.tight_layout()
plt.savefig(
    out_path,
    format='tiff',
    dpi=300,
    bbox_inches='tight'
)

plt.show()


/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/2585522700.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/2585522700.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/2585522700.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/2585522700.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated a

In [25]:
### Fisher ratio ##
sites = ['TS_Ph1b', 'TS_Ph6b']
fisher_dict = {}

for feature in features:
    g1 = df_winter[df_winter['site'] == sites[0]][feature].dropna()
    g2 = df_winter[df_winter['site'] == sites[1]][feature].dropna()
    
    mu1, mu2 = g1.mean(), g2.mean()
    var1, var2 = g1.var(ddof=1), g2.var(ddof=1)
    
    fisher = (mu1 - mu2)**2 / (var1 + var2)
    fisher_dict[feature] = fisher

plt.figure(figsize=(18, 12))

for i, feature in enumerate(features):
    ax = plt.subplot(3, 3, i + 1)
    
    sns.boxplot(
        data=df_winter,
        x='site',
        y=feature,
        showfliers=False,
        width=0.6,
        palette=palette,
        boxprops=dict(facecolor='lightgrey', alpha=0.5),
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(color='black', linewidth=1.5)
    )

    plt.xlabel('')
    plt.ylabel(feature, fontsize=20)
    plt.xticks(ticks=[0, 1], labels=['TS/Ph1b', 'TS/Ph6b'], fontsize=20)
    plt.yticks(fontsize=20)

    # ✅ add Fisher ratio text
    fr = fisher_dict[feature]
    ax.text(
        0.45, 0.94,
        f'FDR = {fr:.2f}',
        transform=ax.transAxes,
        ha='center',
        va='top',
        fontsize=22,
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'),
        zorder=0  # <-- put behind the boxplot
    )

plt.tight_layout()
plt.savefig(
    out_path,
    format='tiff',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/2357567545.py:20: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/2357567545.py:20: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/2357567545.py:20: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_18340/2357567545.py:20: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated a

In [26]:
average_fisher = np.mean(list(fisher_dict.values()))
print(f'Average Fisher ratio across all features: {average_fisher:.2f}')

Average Fisher ratio across all features: 2.30


In [143]:
## Plot for two sites among the shark river slough

out_path = '/Users/bz294/Documents/WhiteZone/Validation/Sawgrass/Winter_Landsat_Boxplot_SRS1c.tif'
# Ensure date is datetime
df_landsat_valid['date'] = pd.to_datetime(df_landsat_valid['date'], errors='coerce')

# Extract month
df_landsat_valid['Month'] = df_landsat_valid['date'].dt.month

# Winter months: Dec–Mar
df_winter = df_landsat_valid[
    df_landsat_valid['Month'].isin([12, 1, 2, 3])
].copy()

# Keep only the two sites
df_winter = df_winter[df_winter['site'].isin(['SRS1b', 'SRS1d'])]
features = ['Blue','Green','Red','NIR','SWIR1','SWIR2','NDVI','NDMI','SAVI']

plt.figure(figsize=(18, 12))

for i, feature in enumerate(features):
    plt.subplot(3, 3, i + 1)

    # Boxplot (median shown by default)
    sns.boxplot(
        data=df_winter,
        x='site',
        y=feature,
        showfliers=False
    )

    # Compute median and std for overlay
    stats = (
        df_winter
        .groupby('site')[feature]
        .agg(['median', 'std'])
        .reset_index()
    )

    # Overlay std as error bars
    plt.errorbar(
        x=range(len(stats)),
        y=stats['median'],
        yerr=stats['std'],
        fmt='o',
        color='black',
        capsize=6,
        linewidth=2
    )

    plt.xlabel('')
    plt.ylabel(feature, fontsize=16)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

plt.tight_layout()
plt.savefig(
    out_path,
    format='tiff',
    dpi=300,
    bbox_inches='tight'
)

plt.show()


/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_46410/485792057.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
